# **CRAWLING 200 BERITA DARI DETIK.COM**

1. Berita Sport = 100 berita
2. Berita Finance = 100 berita

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import re
import sys
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import random


# ============================================================
# 1. PROGRESS BAR
# ============================================================

def print_progress(kategori, current, total):
    """Menampilkan progress crawling."""
    percent = (current / total) * 100 if total > 0 else 0

    bar_length = 20
    filled_length = int(bar_length * current // total) if total > 0 else 0

    bar = '█' * filled_length + '-' * (bar_length - filled_length)

    sys.stdout.write(
        f'\r{kategori} - {current}/{total} '
        f'[{bar}] {percent:.2f}%'
    )
    sys.stdout.flush()

    if current >= total:
        sys.stdout.write('\n')


# ============================================================
# 2. SESSION + RETRY
# ============================================================

def get_session():
    """Membuat session requests dengan mekanisme retry."""

    session = requests.Session()

    retry_strategy = Retry(
        total=5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        backoff_factor=1
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


# ============================================================
# 3. MENGAMBIL JUDUL ARTIKEL
# ============================================================

def get_article_title(soup):
    """Mengambil judul artikel."""

    title_tag = soup.find("h1", class_="detail-title")

    if title_tag:
        return title_tag.get_text(" ", strip=True)

    title_tag = soup.find("h2", class_="media__title")

    if title_tag:
        return title_tag.get_text(" ", strip=True)

    title_tag = soup.find("title")

    if title_tag:
        title = title_tag.get_text(" ", strip=True)

        title = re.sub(
            r"\s*-\s*detik(news|finance|sport|com).*",
            "",
            title,
            flags=re.IGNORECASE
        )

        return title.strip()

    return "Judul Tidak Ditemukan"


# ============================================================
# 4. MENGAMBIL ISI ARTIKEL
# ============================================================

def get_article_content_and_title(session, url):

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/108.0.0.0 Safari/537.36"
        )
    }

    try:

        response = session.get(
            url,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        title = get_article_title(soup)

        # Selector konten artikel
        content_selectors = [
            "div.detail-konten",
            "div.news-detail__content",
            "div.itp_bodycontent",
            "div.content-text",
            "div.article-content",
            "div.text_area"
        ]

        paragraphs = []

        # Cari konten berdasarkan selector
        for selector in content_selectors:

            content_divs = soup.select(selector)

            if content_divs:

                for div in content_divs:

                    for p in div.find_all("p"):

                        text = p.get_text(" ", strip=True)

                        if (
                            text
                            and not text.lower().startswith("baca juga")
                        ):
                            paragraphs.append(text)

                if paragraphs:
                    break

        # Jika selector utama tidak ditemukan
        if not paragraphs:

            article = soup.find("article")

            if article:

                for p in article.find_all("p"):

                    text = p.get_text(" ", strip=True)

                    if (
                        text
                        and not text.lower().startswith("baca juga")
                    ):
                        paragraphs.append(text)

        content = " ".join(paragraphs)

        return title, content

    except requests.exceptions.RequestException as e:

        print(
            f"\n❌ Error fetching {url}: {e}",
            file=sys.stderr
        )

        return "Judul Tidak Ditemukan", ""


# ============================================================
# 5. MENGAMBIL ID BERITA
# ============================================================

def extract_id(url):
    """Mengambil ID berita dari URL."""

    # Format /d-123456
    match = re.search(r"/d-(\d+)", url)

    if match:
        return match.group(1)

    # Angka di akhir URL
    match = re.search(r"-(\d+)$", url)

    if match:
        return match.group(1)

    # Format angka.html
    match = re.search(r"(\d+)\.html$", url)

    if match:
        return match.group(1)

    return None


# ============================================================
# 6. CRAWLING BERITA
# ============================================================

def berita():

    start_time = time.time()

    session = get_session()

    all_articles_data = []

    processed_links = set()

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/108.0.0.0 Safari/537.36"
        )
    }

    # ========================================================
    # TARGET DATA
    # ========================================================

    target_per_category = 100

    categories = {
        "sport": {
            "url": "https://sport.detik.com/indeks",
            "target": 100
        },

        "finance": {
            "url": "https://finance.detik.com/indeks",
            "target": 100
        }
    }

    # ========================================================
    # LOOP SETIAP KATEGORI
    # ========================================================

    for category, config in categories.items():

        base_url = config["url"]
        target = config["target"]

        category_count = 0
        page = 1

        print("\n")
        print("=" * 60)
        print(f"MEMULAI CRAWLING: {category.upper()}")
        print(f"TARGET: {target} BERITA")
        print("=" * 60)

        # Terus mencari halaman sampai mendapatkan 100 artikel
        while category_count < target:

            url = f"{base_url}?page={page}"

            print(
                f"\nMengakses halaman {page}: {url}"
            )

            try:

                response = session.get(
                    url,
                    headers=headers,
                    timeout=15
                )

                response.raise_for_status()

                soup = BeautifulSoup(
                    response.text,
                    "html.parser"
                )

                article_links = soup.select(
                    "a.media__link"
                )

                # Jika tidak ada artikel
                if not article_links:

                    print(
                        f"\n⚠️ Tidak ditemukan artikel "
                        f"pada halaman {page}"
                    )

                    page += 1
                    continue

                print(
                    f"Ditemukan {len(article_links)} link artikel."
                )

                # =================================================
                # LOOP LINK ARTIKEL
                # =================================================

                for a in article_links:

                    # Jika sudah mencapai target
                    if category_count >= target:
                        break

                    link = a.get("href")

                    if not link:
                        continue

                    # Hindari URL duplikat
                    if link in processed_links:
                        continue

                    processed_links.add(link)

                    # Ambil ID
                    berita_id = extract_id(link)

                    # Ambil judul dan isi
                    title, content = (
                        get_article_content_and_title(
                            session,
                            link
                        )
                    )

                    # Hanya masukkan jika konten berhasil
                    if content:

                        category_count += 1

                        print_progress(
                            category,
                            category_count,
                            target
                        )

                        all_articles_data.append({

                            "id_berita": berita_id,

                            "judul_berita": title,

                            "isi_berita_original": content,

                            "kategori_berita": category,

                            "url_berita": link
                        })

                    # Jeda random
                    time.sleep(
                        random.uniform(1, 2)
                    )

            except requests.exceptions.RequestException as e:

                print(
                    f"\n❌ Gagal mengakses {url}: {e}"
                )

                # Tunggu sebelum mencoba halaman berikutnya
                time.sleep(3)

            page += 1

        print(
            f"\n✅ {category.upper()} selesai: "
            f"{category_count} berita"
        )

    # ============================================================
    # MEMBUAT DATAFRAME
    # ============================================================

    df = pd.DataFrame(all_articles_data)

    # ============================================================
    # VALIDASI DATA
    # ============================================================

    print("\n")
    print("=" * 60)
    print("HASIL AKHIR CRAWLING")
    print("=" * 60)

    print(
        "\nJumlah berita berdasarkan kategori:"
    )

    print(
        df["kategori_berita"].value_counts()
    )

    print(
        f"\nTotal berita: {len(df)}"
    )

    # ============================================================
    # SIMPAN CSV
    # ============================================================

    output_file = "crawling_detik_sport_finance_200.csv"

    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    # ============================================================
    # WAKTU EKSEKUSI
    # ============================================================

    end_time = time.time()

    elapsed = int(
        end_time - start_time
    )

    jam, sisa = divmod(
        elapsed,
        3600
    )

    menit, detik = divmod(
        sisa,
        60
    )

    print(
        f"\n💾 File berhasil disimpan:"
        f"\n{output_file}"
    )

    print(
        f"\n⏱️ Waktu eksekusi:"
        f" {jam} jam {menit} menit {detik} detik"
    )

    print("\n5 data pertama:")

    print(
        df.head()
    )

    return df


# ============================================================
# 7. JALANKAN PROGRAM
# ============================================================

if __name__ == "__main__":

    df = berita()



MEMULAI CRAWLING: SPORT
TARGET: 100 BERITA

Mengakses halaman 1: https://sport.detik.com/indeks?page=1


Ditemukan 41 link artikel.


sport - 1/100 [--------------------] 1.00%

sport - 2/100 [--------------------] 2.00%

sport - 3/100 [--------------------] 3.00%

...

sport - 6/100 [█-------------------] 6.00%

sport - 7/100 [█-------------------] 7.00%

sport - 8/100 [█-------------------] 8.00%

sport - 9/100 [█-------------------] 9.00%

KeyboardInterrupt: 

In [2]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Gunakan data hasil crawling yang sudah ada di DataFrame.
if "df" not in globals():
    df = pd.read_csv("crawling_detik_sport_finance200.csv")

kolom_teks = "isi_berita_original"
if kolom_teks not in df.columns:
    raise KeyError(f"Kolom '{kolom_teks}' tidak ditemukan. Kolom yang tersedia: {list(df.columns)}")

teks_berita = df[kolom_teks].fillna("").astype(str)

# Ambil kata alfabet, ubah menjadi huruf kecil, dan abaikan kata satu huruf.
vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{2,}\b"
)
matriks_kata = vectorizer.fit_transform(teks_berita)
kata_unik = vectorizer.get_feature_names_out()

print(f"Jumlah berita yang dianalisis: {len(teks_berita)}")
print(f"Jumlah kata unik: {len(kata_unik)}")
print("100 kata unik pertama:", list(kata_unik[:100]))

Jumlah berita yang dianalisis: 200
Jumlah kata unik: 7183
100 kata unik pertama: ['aadi', 'aam', 'aan', 'aau', 'abadi', 'abang', 'abdi', 'abdul', 'abdullah', 'aberdeen', 'abraham', 'absen', 'absennya', 'abt', 'abu', 'acara', 'access', 'account', 'accurate', 'aceh', 'achmad', 'acosta', 'action', 'activ', 'activity', 'acuan', 'acuannya', 'ada', 'adalah', 'adanya', 'adaptasi', 'adapun', 'adhi', 'adi', 'adik', 'adil', 'adininggar', 'adk', 'administrasi', 'adrenalin', 'adu', 'aduan', 'advokat', 'aeon', 'aeromodelling', 'af', 'afiliasi', 'agama', 'agar', 'agen', 'agency', 'agenda', 'agennya', 'agent', 'agraria', 'agreement', 'agresif', 'agrinas', 'agung', 'agus', 'agustus', 'agustusan', 'ahhn', 'ahlinya', 'ahmad', 'ahren', 'ai', 'aichi', 'air', 'airin', 'airlangga', 'airport', 'aja', 'ajaib', 'ajak', 'ajakan', 'ajaknya', 'ajang', 'aji', 'ajukan', 'akademi', 'akademik', 'akademisi', 'akal', 'akan', 'akbar', 'akhir', 'akhiri', 'akhirnya', 'akibat', 'akibatnya', 'aklamasi', 'akm', 'akomodasi', 